In [16]:
using Pkg
Pkg.activate("..")
using Revise
using MajoranaPropagation
include("../src/imaginary_gates.jl")
using PauliPropagation

using Plots


  Activating project at `~/.julia/dev/Sparqle/MajoranaPropagation.jl`


Simulate the dynamics of the spinful Hubbard model on a 2D lattice of $N_x \times N_y$ spinful sites. The Hamiltonian is
$$\hat{H}=-t \sum_{\langle i,j\rangle, \sigma=\{\uparrow, \downarrow\}}\left(\hat{c}_{i, \sigma}^{\dagger} \hat{c}_{j, \sigma}+\hat{c}_{j, \sigma}^{\dagger} \hat{c}_{i, \sigma}\right)+U \sum_i \hat{n}_{i \uparrow} \hat{n}_{i \downarrow}$$

In [49]:
N_x = 2
N_y = 2
N_spinful_sites = N_x * N_y
t = 1.
U = 2.

n_layers = 12
dt = 0.07

0.07

Get 1D connectivity, and create the circtuit for implementing a single layer of first order Trotterization.

In [3]:
topo = rectangletopology(N_x, N_y)

circ_single = []
thetas_single = []

#up hoppings 
for (i, j) in topo
    push!(circ_single, FermionicGate(:hopup, [i, j]))
    push!(thetas_single, -t * dt)
end

#down hoppings 
for (i, j) in topo
    push!(circ_single, FermionicGate(:hopdn, [i, j]))
    push!(thetas_single, -t * dt)
end

#on-site repulsion 
for i = 1:N_spinful_sites
    push!(circ_single, FermionicGate(:nupndn, i))
    push!(thetas_single, U * dt)
end

#### Set the truncations
1. `min_abs_coeff`: PP coefficient truncation
2. `max_singles`: number of unpaired Majoranas, namely the number of indices $i$ where only one of $\gamma_i$, $\gamma'_i$ are non-zero

In [4]:
min_abs_coeff = 1e-5
max_unpaired = 6

# max_single_filter = create_max_single_filter(2 * N_spinful_sites)
# custom_trunc = let max_single_filter=max_single_filter, max_singles = max_singles
#     (mstr, coeff) -> (compute_max_single(mstr, 0, max_single_filter) > max_singles)
# end

6

Set the inital state as the checkerboard state $\ket{\uparrow\downarrow\cdots\uparrow\downarrow}$

In [5]:
#initial state 
initial_state_label = "Checkerboard"

create_up_part_at = []
create_down_part_at = []

for j=1:N_spinful_sites 
    if j % 2 == 1
        push!(create_up_part_at, j)
    else 
        push!(create_down_part_at, j)
    end 
end

Backpropagate $n_{2,\uparrow}$, the up density on site 2

In [6]:
# to = TimerOutput()

site_index = 3
obs = MajoranaSum(N_spinful_sites, :nupndn, site_index)
obs = VectorMajoranaSum(obs)

@show obs 

res = zeros(n_layers+1)
# res[1] = overlap_with_fock_spinful(obs, create_up_part_at, create_down_part_at, 2 * N_spinful_sites)
length_res = zeros(n_layers+1)
length_res[1] = length(obs)

for k=1:n_layers-1
    obs = propagate!(circ_single, obs, thetas_single; min_abs_coeff, max_unpaired)
    # res[k+1] = overlap_with_fock_spinful(obs, create_up_part_at, create_down_part_at, 2 * N_spinful_sites)
    length_res[k+1] = length(obs)
end 


obs = VectorMajoranaSum with 4 terms:
0.25 * 0000000000000000000000000000000000000000000000000000000000000000
0.25 * 0000000011000000000000000000000000000000000000000000000000000000
-0.25 * 0000000011110000000000000000000000000000000000000000000000000000
0.25 * 0000000000110000000000000000000000000000000000000000000000000000



In [11]:
using ProfileCanvas
ProfileCanvas.@profview propagate(circ_single, obs, thetas_single; min_abs_coeff, max_unpaired)

ProfileCanvas.ProfileData(Dict{String, ProfileCanvas.ProfileFrame}("4" => ProfileCanvas.ProfileFrame("root", "", "", 0, 432, missing, 0x00, missing, ProfileCanvas.ProfileFrame[ProfileCanvas.ProfileFrame("task_done_hook", "task.jl", "./task.jl", 868, 266, missing, 0x10, missing, ProfileCanvas.ProfileFrame[ProfileCanvas.ProfileFrame("wait", "task.jl", "./task.jl", 1228, 266, missing, 0x10, missing, ProfileCanvas.ProfileFrame[ProfileCanvas.ProfileFrame("poptask", "task.jl", "./task.jl", 1216, 266, missing, 0x11, missing, ProfileCanvas.ProfileFrame[ProfileCanvas.ProfileFrame("trypoptask", "task.jl", "./task.jl", 1208, 6, missing, 0x10, missing, ProfileCanvas.ProfileFrame[ProfileCanvas.ProfileFrame("multiq_deletemin", "partr.jl", "./partr.jl", 191, 1, missing, 0x00, missing, ProfileCanvas.ProfileFrame[ProfileCanvas.ProfileFrame("getindex", "abstractarray.jl", "./abstractarray.jl", 1345, 1, missing, 0x00, missing, ProfileCanvas.ProfileFrame[ProfileCanvas.ProfileFrame("getindex", "essentials.jl", "./essentials.jl", 919, 1, missing, 0x00, missing, ProfileCanvas.ProfileFrame[ProfileCanvas.ProfileFrame("length", "essentials.jl", "./essentials.jl", 11, 1, missing, 0x00, missing, ProfileCanvas.ProfileFrame[])])])]), ProfileCanvas.ProfileFrame("multiq_deletemin", "partr.jl", "./partr.jl", 186, 1, missing, 0x00, missing, ProfileCanvas.ProfileFrame[ProfileCanvas.ProfileFrame("==", "promotion.jl", "./promotion.jl", 637, 1, missing, 0x00, missing, ProfileCanvas.ProfileFrame[])]), ProfileCanvas.ProfileFrame("multiq_deletemin", "partr.jl", "./partr.jl", 193, 1, missing, 0x00, missing, ProfileCanvas.ProfileFrame[ProfileCanvas.ProfileFrame(">", "operators.jl", "./operators.jl", 425, 1, missing, 0x00, missing, ProfileCanvas.ProfileFrame[ProfileCanvas.ProfileFrame("<", "int.jl", "./int.jl", 519, 1, missing, 0x00, missing, ProfileCanvas.ProfileFrame[])])]), ProfileCanvas.ProfileFrame("multiq_deletemin", "partr.jl", "./partr.jl", 196, 1, missing, 0x00, missing, ProfileCanvas.ProfileFrame[]), ProfileCanvas.ProfileFrame("multiq_deletemin", "partr.jl", "./partr.jl", 192, 1, missing, 0x00, missing, ProfileCanvas.ProfileFrame[ProfileCanvas.ProfileFrame("getindex", "abstractarray.jl", "./abstractarray.jl", 1345, 1, missing, 0x00, missing, ProfileCanvas.ProfileFrame[ProfileCanvas.ProfileFrame("getindex", "essentials.jl", "./essentials.jl", 919, 1, missing, 0x00, missing, ProfileCanvas.ProfileFrame[])])]), ProfileCanvas.ProfileFrame("multiq_deletemin", "partr.jl", "./partr.jl", 172, 1, missing, 0x00, missing, ProfileCanvas.ProfileFrame[])]), ProfileCanvas.ProfileFrame("trypoptask", "task.jl", "./task.jl", 1194, 1, missing, 0x10, missing, ProfileCanvas.ProfileFrame[]), ProfileCanvas.ProfileFrame("multiq_check_empty", "partr.jl", "./partr.jl", 237, 1, missing, 0x00, missing, ProfileCanvas.ProfileFrame[])])])]), ProfileCanvas.ProfileFrame("#itask_partition##0", "task_partitioner.jl", "/home/manuel/.julia/packages/AcceleratedKernels/AdYRJ/src/task_partitioner.jl", 260, 84, missing, 0x00, missing, ProfileCanvas.ProfileFrame[ProfileCanvas.ProfileFrame("#47", "cpu_sample_sort.jl", "/home/manuel/.julia/packages/AcceleratedKernels/AdYRJ/src/sort/cpu_sample_sort.jl", 135, 48, missing, 0x00, missing, ProfileCanvas.ProfileFrame[ProfileCanvas.ProfileFrame("_sample_sort_sort_bucket!", "cpu_sample_sort.jl", "/home/manuel/.julia/packages/AcceleratedKernels/AdYRJ/src/sort/cpu_sample_sort.jl", 73, 48, missing, 0x00, missing, ProfileCanvas.ProfileFrame[ProfileCanvas.ProfileFrame("#_sample_sort_sort_bucket!#38", "cpu_sample_sort.jl", "/home/manuel/.julia/packages/AcceleratedKernels/AdYRJ/src/sort/cpu_sample_sort.jl", 92, 26, missing, 0x00, missing, ProfileCanvas.ProfileFrame[ProfileCanvas.ProfileFrame("sort!", "sort.jl", "./sort.jl", 1733, 26, missing, 0x00, missing, ProfileCanvas.ProfileFrame[ProfileCanvas.ProfileFrame("#sort!#23", "sort.jl", "./sort.jl", 1740, 26, missing, 0x00, missing, ProfileCanvas.ProfileFrame[ProfileCanvas.ProfileFrame("_sort!", "sort.jl", "./sort.jl", 15

### Imaginary rotation

In [105]:
imag_layer = [ImaginaryFermionicGate(:nup, i) for i in 1:N_spinful_sites]  # imaginary time evolution layer
imag_thetas = [-U * 0.05 for i in 1:N_spinful_sites]

4-element Vector{Float64}:
 -0.1
 -0.1
 -0.1
 -0.1

In [106]:
H = MajoranaSum(N_spinful_sites, :nup, 1)
for i in 2:N_spinful_sites
    H += MajoranaSum(N_spinful_sites, :nup, i)
end
H

MajoranaSum with 5 term(s):(
    2.0 * 0000000000000000
    0.5 * 0000000011000000
    0.5 * 0000000000001100
    0.5 * 1100000000000000
    0.5 * 0000110000000000)

In [107]:
obs = MajoranaSum(N_spinful_sites, true, Dict(getinttype(N_spinful_sites*2)(0) => 1.0))
obs = VectorMajoranaSum(obs)

VectorMajoranaSum with 1 term:
1.0 * 0000000000000000


In [108]:
include("../src/imaginary_gates.jl")

scalarproduct (generic function with 1 method)

In [109]:
for _ in 1:30
    obs = propagate!(imag_layer, obs, imag_thetas; min_abs_coeff=0)
    obs.coeffs ./= getcoeff(obs, 0)
    println(scalarproduct(H, obs))
end

1.8006640107500882
1.601288677234002
1.4015195280109873
1.2003666273764972
0.9958580255376642
0.7846120688769769
0.5612118505644892
0.31714482486756734
0.03880434324266269
-0.29662059244001576
-0.7304647054998272
-1.346684109322345
-2.347497784672308
-4.37961064397267
-11.270198986291982
54.71313747009779
9.501069348249153
5.581984778485783
4.09543511603165
3.2995121441494915
2.795105649370387
2.440866290857457
2.1739101325418737
1.961905385281157
1.7864419684793136
1.6361886205802685
1.5037125770293864
1.3838567262804362
1.2728454053736717
1.1677563078278794


In [101]:
obs

VectorMajoranaSum with 16 terms:
1.0 * 0000000000000000
-0.9998844933954714 * 1100000000000000
-0.9998844942853449 * 0000110000000000
-0.9997690010018481 * 1100110000000000
6.104979273008148 * 0000000011000000
6.1042746308739035 * 1100000011000000
6.104274630764383 * 0000110011000000
-6.103570069963108 * 1100110011000000
6.1049792730081505 * 0000000000001100
6.1042746308739035 * 1100000000001100
6.1042746307643805 * 0000110000001100
-6.103570069963103 * 1100110000001100
-37.257643128908214 * 0000000011001100
37.253342754240315 * 1100000011001100
37.25334275425379 * 0000110011001100
37.24904287594606 * 1100110011001100
